# CONUS404: Monthly Precipitation over the Contiguous US

```{image} ../../thumbnails/gdex_logo.png
:alt: GDEX Cookbook logo
:width: 200px
```

---

## Overview

CONUS404 is a 4-km, hourly, 43-year hydroclimate reanalysis of the contiguous
United States, produced by NCAR and the USGS by dynamically downscaling ERA5 with
the WRF model in a convection-permitting configuration
([Rasmussen et al., 2023](https://journals.ametsoc.org/view/journals/bams/104/8/BAMS-D-21-0326.1.xml)).
GDEX hosts it as dataset [`d559000`](https://gdex.ucar.edu/datasets/d559000/).

The native data is ~377,000 hourly NetCDF files; GDEX exposes them through a single
[kerchunk](https://fsspec.github.io/kerchunk/) reference, so we can slice the record
with one `xarray` call and stream only the bytes we need — here, over HTTPS.

We map total precipitation for a **single month** over CONUS — a scaled-down view
of the multi-year climatology in Rasmussen et al. (2023, Fig. 3), kept light enough
to run interactively.

1. Open CONUS404 2D output from a kerchunk reference over HTTPS
2. Select one month of hourly precipitation
3. Sum to a monthly total
4. Map it over CONUS on the model's Lambert Conformal grid

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [ERA5 Reanalysis Data Workflow](era5.ipynb) | Helpful | The kerchunk reference access pattern |
| [xarray](https://docs.xarray.dev) | Necessary | Labeled N-D arrays |
| [Dask](https://docs.dask.org) | Necessary | Parallel / out-of-core compute |
| [Cartopy](https://scitools.org.uk/cartopy) | Necessary | Mapping |

- **Time to learn**: 30 minutes

---

## Imports

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm
from dask.distributed import LocalCluster

## Spin up a Dask cluster

Summing a month of hourly 4-km precipitation streams a few GB of chunks. A
`LocalCluster` reads them in parallel and keeps memory bounded.

In [2]:
cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=2,
    memory_limit="4GiB",  # per worker
)
client = cluster.get_client()
client

/srv/conda/envs/notebook/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44717 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:44717/status,
Dashboard: http://127.0.0.1:44717/status,Workers: 4
Total threads: 8,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42653,Workers: 0
Dashboard: http://127.0.0.1:44717/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:45823,Total threads: 2
Dashboard: http://127.0.0.1:45145/status,Memory: 4.00 GiB
Nanny: tcp://127.0.0.1:42271,


## CONUS404 on GDEX

GDEX publishes a kerchunk reference for all **2D** CONUS404 variables (`wrf2d`) —
precipitation, 2-m temperature, surface fluxes, and more. We open it over HTTPS and
pull out hourly accumulated precipitation, `PREC_ACC_NC` (mm), along with the static
2D lat/lon needed for mapping.

In [4]:
kerchunk_url = "https://data.gdex.ucar.edu/d559000/kerchunk/wrf2d-remote-osdf.parq"

ds = xr.open_dataset(kerchunk_url, engine="kerchunk")
pr = ds["PREC_ACC_NC"]
lat2d = ds["XLAT"].isel(Time=0)
lon2d = ds["XLONG"].isel(Time=0)
pr

TypeError: boolean value of NA is ambiguous